# House Price Prediction: Exploratory Data Analysis (EDA)

Idhu EDA notebook. Project folder-la `notebooks/eda.ipynb` nu save pannunga (illa project root-la). Dataset: `data/train.csv` (Ames housing).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

# Works whether notebook is in project root or in notebooks/ folder
DATA_PATH = "data/train.csv" if os.path.exists("data/train.csv") else "../data/train.csv"
PLOT_DIR = "eda_plots" if DATA_PATH == "data/train.csv" else "../eda_plots"
os.makedirs(PLOT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
TARGET = "SalePrice"
print("Shape:", df.shape)
df.head()

## 1. Dataset overview

In [ ]:
df.info()

In [ ]:
# Numeric summary (mean, std, min, max...)
df.describe().T

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Numeric columns:", df.select_dtypes(include=np.number).shape[1])
print("Categorical columns:", df.select_dtypes(exclude=np.number).shape[1])

## 2. Missing values

`Alley`, `PoolQC`, `Fence`, `MiscFeature` la romba values missing, so project-la andha 4 columns drop pannom.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = (pd.DataFrame({"missing_count": missing, "missing_%": missing_pct})
              .query("missing_count > 0")
              .sort_values("missing_%", ascending=False))
display(missing_df)

plt.figure(figsize=(10, 6))
sns.barplot(x=missing_df["missing_%"], y=missing_df.index, color="steelblue")
plt.axvline(50, color="red", linestyle="--", label="50% line")
plt.title("Missing Values by Column (%)")
plt.xlabel("Missing %")
plt.legend()
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/01_missing_values.png", dpi=150)
plt.show()

## 3. Target variable: SalePrice (histogram)

Price right-skewed (konjam periya expensive veedugal). Adhanaala model-ku munnadi **log transform** panninom.

In [ ]:
print("Skewness (raw):", round(df[TARGET].skew(), 3))
print("Skewness (log):", round(np.log1p(df[TARGET]).skew(), 3))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(df[TARGET], kde=True, bins=40, color="steelblue", ax=axes[0])
axes[0].set_title("SalePrice (original)")
sns.histplot(np.log1p(df[TARGET]), kde=True, bins=40, color="seagreen", ax=axes[1])
axes[1].set_title("SalePrice (log transformed)")
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/02_saleprice_histogram.png", dpi=150)
plt.show()

## 4. Histograms of important numeric features

In [ ]:
num_cols = ["GrLivArea", "LotArea", "TotalBsmtSF", "GarageArea", "YearBuilt", "OverallQual"]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(df[col].dropna(), kde=True, bins=40, ax=ax, color="steelblue")
    ax.set_title(col)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/03_numeric_histograms.png", dpi=150)
plt.show()

## 5. Correlation heatmap

SalePrice-oda adhigama correlate aagura features edhu nu paakalam.

In [ ]:
corr = df.select_dtypes(include=np.number).corr()

# Top 12 features most correlated with SalePrice
top = corr[TARGET].abs().sort_values(ascending=False).head(12).index
plt.figure(figsize=(11, 9))
sns.heatmap(df[top].corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True, linewidths=0.5)
plt.title("Correlation Heatmap: Top 12 Features with SalePrice")
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/04_correlation_heatmap.png", dpi=150)
plt.show()

print("Top correlations with SalePrice:")
corr[TARGET].drop(TARGET).sort_values(ascending=False).head(10).round(3)

## 6. SalePrice vs GrLivArea (outliers)

Peria GrLivArea irundhum price kammi-a irukkura points = outliers. Project-la IQR method (factor 3.0) la remove pannom.

In [ ]:
def iqr_outlier_mask(series, factor=3.0):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return (series < q1 - factor * iqr) | (series > q3 + factor * iqr)

out_gr = iqr_outlier_mask(df["GrLivArea"])
print("GrLivArea outliers:", out_gr.sum())

plt.figure(figsize=(9, 6))
plt.scatter(df.loc[~out_gr, "GrLivArea"], df.loc[~out_gr, TARGET], alpha=0.5, label="Normal", color="steelblue")
plt.scatter(df.loc[out_gr, "GrLivArea"], df.loc[out_gr, TARGET], alpha=0.9, label="Outlier", color="red")
sns.regplot(x="GrLivArea", y=TARGET, data=df[~out_gr], scatter=False, color="black", line_kws={"linewidth": 1.5})
plt.title("SalePrice vs GrLivArea")
plt.xlabel("GrLivArea (sq ft)")
plt.ylabel("SalePrice")
plt.legend()
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/05_price_vs_grlivarea.png", dpi=150)
plt.show()

## 7. SalePrice vs OverallQual

In [ ]:
plt.figure(figsize=(9, 6))
sns.boxplot(x="OverallQual", y=TARGET, data=df, palette="viridis", hue="OverallQual", legend=False)
plt.title("SalePrice by OverallQual")
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/06_price_vs_overallqual.png", dpi=150)
plt.show()

## 8. SalePrice by Neighborhood

Neighborhood-ku price romba maaruthu. Adhanaala **target encoding** use pannom.

In [ ]:
order = df.groupby("Neighborhood")[TARGET].median().sort_values(ascending=False).index
plt.figure(figsize=(12, 6))
sns.boxplot(x="Neighborhood", y=TARGET, data=df, order=order, color="lightsteelblue")
plt.xticks(rotation=60)
plt.title("SalePrice by Neighborhood (sorted by median)")
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/07_price_by_neighborhood.png", dpi=150)
plt.show()

## 9. Outlier analysis: boxplots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ["GrLivArea", "LotArea", TARGET]):
    sns.boxplot(y=df[col], ax=ax, color="lightcoral")
    ax.set_title(f"Boxplot: {col}")
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/08_outlier_boxplots.png", dpi=150)
plt.show()

for col in ["GrLivArea", "LotArea"]:
    print(f"{col}: {iqr_outlier_mask(df[col]).sum()} outliers (IQR factor 3.0)")

## 10. Key findings (summary)

- **SalePrice** right-skewed, so log transform use pannom.
- **OverallQual** and **GrLivArea** price-oda strongest correlation.
- **Alley, PoolQC, Fence, MiscFeature** la romba missing values, so drop pannom.
- **GrLivArea** and **LotArea** la outliers irukku, remove pannom.
- **Neighborhood** price-ai romba affect pannum, so target-encode pannom.

Saved plots: `eda_plots/` folder la irukkum (assignment report-la use pannalam).